In [1]:
# Modules requires
import hail as hl
import os

from gnomad.utils.vep import process_consequences
from ukb_utils import hail_init
from ko_utils import qc

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('/logs/hail/hail_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe055.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to /logs/hail/hail_export.log


In [3]:
dataset = qc.get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
#result = hl.vep(dataset, "utils/configs/vep_env.json") 
#result = process_consequences(result)

In [9]:
### now moved to QC module!

def annotate_snpid(mt, delim = '_'):
    r'''Annotate field snpid based on current locus and alleles'''
    ids = hl.delimit([mt.locus.contig, hl.str(mt.locus.position), mt.alleles[0], mt.alleles[1]], delim)
    mt = mt.annotate_rows(snpid = ids)
    return mt

def annotate_rsid(mt, dbsnp_path = '/well/lindgren/flassen/ressources/dbsnp/GRCh38/GCF_000001405.39.gz', build = 'GRCh38'):
    r'''Use dbSNP to annotate all rsIDs in the a matrix table.'''
    recode = {f"NC_0000{i}.{j}":f"chr{i}" for i in (list(range(1, 23)) + ['X', 'Y']) for j in ('09','10','11','12','13','14')}
    dbsnp = hl.import_vcf(dbsnp_path, 
                       reference_genome=build, 
                       contig_recoding=recode, 
                       skip_invalid_loci=True,
                       force_bgz=True)
    
    rsids = dbsnp.index_rows(mt.locus, mt.alleles).rsid
    mt = mt.annotate_rows(rsid = rsids)
    return mt


def default_to_snpid_when_missing_rsid(mt):
    r'''rsid is converted to snpid when it is missing'''
    return mt.annotate_rows(rsid = hl.if_else(hl.is_missing(mt.rsid), mt.snpid, mt.rsid))




In [ ]:
def annotate_vep(mt, vep_path):
    r'''Annotate matrix table with VEP consequence from external file.'''
    print(f'Annotating with VEP file: {vep_path}')
    
    # Open file containing VEP fields
    with open('data/vep/vep_fields.txt', 'r') as file:
        fields = file.read().strip().split(',')
    ht = hl.import_vcf(vep_path).rename({'info':'vep'}) 
    
    # Add VEP fields by iteration
    for i in range(len(fields)):
        ht = ht.annotate_rows(
            vep=ht.vep.annotate(
                col=ht.vep.CSQ.map(lambda x: (x.split('\\|')[i]))[0]
                ).rename({'col':f'{fields[i]}'})
        )
    
    # Most severe variant consequence
    ht = ht.annotate_rows(vep = ht.vep.annotate(most_severe_consequence = ht.vep.Consequence.split('&')[0]))
    
    # Extract various categories annotations and change type
    ht = ht.annotate_rows(vep = ht.vep.annotate(sift_pred = ht.vep.SIFT_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hdiv_pred = ht.vep.Polyphen2_HDIV_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hvar_pred = ht.vep.Polyphen2_HVAR_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(cadd_phred_score = hl.parse_float(ht.vep.CADD_phred)))
    ht = ht.annotate_rows(vep = ht.vep.annotate(revel_score = hl.parse_float(ht.vep.REVEL_score)))
    
    # Define protein truncating variants
    ptv = hl.set(["transcript_ablation", "splice_acceptor_variant",
              "splice_donor_variant", "stop_gained", "frameshift_variant"])
    
    # Define missense variation
    missense = hl.set(["stop_lost", "start_lost", "transcript_amplification",
                   "inframe_insertion", "inframe_deletion", "missense_variant",
                   "protein_altering_variant", "splice_region_variant"])
    
    # Define synonymous
    synonymous = hl.set(["incomplete_terminal_codon_variant", "stop_retained_variant", "synonymous_variant"])
    
    # Define non coding variation
    non_coding = hl.set(["coding_sequence_variant", "mature_miRNA_variant", "5_prime_UTR_variant",
              "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant",
              "NMD_transcript_variant", "non_coding_transcript_variant", "upstream_gene_variant",
              "downstream_gene_variant", "TFBS_ablation", "TFBS_amplification", "TF_binding_site_variant",
              "regulatory_region_ablation", "regulatory_region_amplification", "feature_elongation",
              "regulatory_region_variant", "feature_truncation", "intergenic_variant"])
    
    # Create categories for downstream analysis
    ht = ht.annotate_rows(vep = ht.vep.annotate(consequence_category = 
        hl.case().when(ptv.contains(ht.vep.most_severe_consequence), "ptv")
             .when(missense.contains(ht.vep.most_severe_consequence) & 
                   (~hl.is_defined(ht.vep.cadd_phred_score) | 
                    ~hl.is_defined(ht.vep.revel_score)), "other_missense")                                   
             .when(missense.contains(ht.vep.most_severe_consequence) & 
                   (ht.vep.cadd_phred_score >= 20) & 
                   (ht.vep.revel_score >= 0.6), "damaging_missense") 
             .when(missense.contains(ht.vep.most_severe_consequence), "other_missense")
             .when(synonymous.contains(ht.vep.most_severe_consequence), "synonymous")
             .when(non_coding.contains(ht.vep.most_severe_consequence), "non_coding")
             .default("NA")
    ))
                                                
    # combine with matrix table
    mt = mt.annotate_rows(vep = ht.index_rows(mt.locus, mt.alleles).vep.drop('CSQ'))
    return(mt)

In [ ]:
def maf_category_case_builder(mt, bins = [0.1, 0.01,0.001,0.0001,0.00001]):
    return (hl.case()
            .when(call_stats_expr.AC <= 5, call_stats_expr.AC)
            .when(call_stats_expr.AC <= 10, 10)
            .when(call_stats_expr.AC <= 20, 20)
            .when(call_stats_expr.AF <= 0.001, 0.001)
            .when(call_stats_expr.AF <= 0.01, 0.01)
            .when(call_stats_expr.AF <= 0.1, 0.1)
            .default(0.99))


def mac_category_case_builder(call_stats_expr):
    return (hl.case()
            .when(call_stats_expr.AC <= 5, call_stats_expr.AC)
            .when(call_stats_expr.AC <= 10, 10)
            .when(call_stats_expr.AC <= 20, 20)
            .when(call_stats_expr.AF <= 0.001, 0.001)
            .when(call_stats_expr.AF <= 0.01, 0.01)
            .when(call_stats_expr.AF <= 0.1, 0.1)
            .default(0.99))
    
    

In [10]:
dataset = annotate_snpid(dataset)
dataset = annotate_rsid(dataset)
dataset = default_to_snpid_when_missing_rsid(dataset)

2021-09-15 18:57:48 Hail: WARN: expected input file 'file:/well/lindgren/flassen/ressources/dbsnp/GRCh38/GCF_000001405.39.gz' to end in .vcf[.bgz, .gz]


In [ ]:
dataset.rsid.show()

2021-09-15 18:46:20 Hail: INFO: Coerced almost-sorted dataset
2021-09-15 18:54:02 Hail: INFO: Coerced almost-sorted dataset


In [20]:
#dbsnp[(dataset.locus, dataset.alleles)].rsid


In [ ]:
import hail as hl
import os

from gnomad.utils.vep import process_consequences
from ukb_utils import hail_init
from ko_utils import qc

def main(args):
    
    # input path
    input_path = args.input_path
    input_type = args.input_type
    out_prefix = args.out_prefix
    out_type = args.out_type
    
    
    # annotate with VEP
    hail_init.hail_bmrc_init('/logs/hail/hail_vep_export.log', 'GRCh38')
    dataset = qc.get_table(input_path, input_type)
    result = hl.vep(dataset, "utils/configs/vep_env.json") 
    result = process_consequences(result)
    qc.export_table(result, out_prefix = out_prefix, out_type = out_type)
    
if __name__=='__main__':
    parser = argparse.ArgumentParser()
    # initial params
    parser.add_argument('--input_path', default=None, help='Path to input')
    parser.add_argument('--input_type', default=None, help='Input type, either "mt", "vcf" or "plink"')
    parser.add_argument('--out_prefix', default=None, help='Path prefix for output dataset')
    parser.add_argument('--out_type', default=None, help='Type of output dataset (options: mt, vcf, plink)')
    args = parser.parse_args()

    main(args)